In [ ]:
# %% [markdown]
# 0) Config — toggles, label mode

# %%
# -------- Global Config (enhanced for 32GB M1, log-only by default) --------
CFG = {
    "RECORD_FILE_COL": "filename_lr",          # 'filename_lr' (100 Hz) or 'filename_hr' (500 Hz)
    "LABEL_MODE": "5class",                    # or '10class'

    # Runtime profile
    "FAST_RUN": False,
    "MAX_RECORDS": None,
    "SEED": 42,

    # Disk/cache toggles (log-only defaults)
    "USE_FEATURE_CACHE": False,                # don't read/write cached CSV
    "DEEP_CACHE_TO_DISK": False,               # keep tf.data cache in RAM (tips will flip to True)
    "SAVE_MIN_TABLE": False,                   # don't write minimal table CSV
    "SAVE_ARTIFACTS": False,                   # don't write models/CSVs/ZIPs

    "ART_DIR": "test/artifacts",
    "EDA_SKIP_HEAVY": True,

    # Memory & device knobs (baseline; profile below will override)
    "LOW_RAM": True,
    "TF_DEVICE": "gpu",
    "TF_MIXED_PRECISION": False,
    "TF_XLA": False,
    "TF_INTRA_OP": 1,
    "TF_INTER_OP": 1,

    # Deep pipeline behavior
    "DEEP_FAST_METADATA_ONLY": True,
    "DEEP_MAX_TRAIN_FRAC": 0.25,

    # Models to run
    "RUN_TF_CNN":  True,
    "RUN_TF_RNN":  True,
    "RUN_TF_LSTM": True,
    "RUN_TF_ANN":  True,
    "DEEP_HYBRID": False,

    # Sequence & training sizes (defaults; profile below will override)
    "SEQ_LEN": 300,
    "DOWNSAMPLE_FACTOR": 5,
    "BATCH_SIZE": 16,
    "EPOCHS": 6,

    # Optional binary task (keep False; multiclass is main)
    "BINARY_TASK": False,
    "BINARY_SCHEME": "NORM_vs_ALL"
}

# Fast-run overrides
if CFG["FAST_RUN"] and (CFG["MAX_RECORDS"] is None):
    CFG["MAX_RECORDS"] = 200
if CFG["FAST_RUN"]:
    CFG["EPOCHS"] = min(CFG["EPOCHS"], 4)

print(CFG)


{'RECORD_FILE_COL': 'filename_lr', 'LABEL_MODE': '5class', 'FAST_RUN': False, 'MAX_RECORDS': None, 'SEED': 42, 'USE_FEATURE_CACHE': False, 'DEEP_CACHE_TO_DISK': False, 'SAVE_MIN_TABLE': False, 'SAVE_ARTIFACTS': False, 'ART_DIR': 'test/artifacts', 'EDA_SKIP_HEAVY': True, 'LOW_RAM': True, 'TF_DEVICE': 'cpu', 'TF_MIXED_PRECISION': False, 'TF_XLA': False, 'TF_INTRA_OP': 1, 'TF_INTER_OP': 1, 'DEEP_FAST_METADATA_ONLY': True, 'DEEP_MAX_TRAIN_FRAC': 0.25, 'RUN_TF_CNN': True, 'RUN_TF_RNN': True, 'RUN_TF_LSTM': True, 'RUN_TF_ANN': True, 'DEEP_HYBRID': False, 'SEQ_LEN': 300, 'DOWNSAMPLE_FACTOR': 5, 'BATCH_SIZE': 16, 'EPOCHS': 6, 'BINARY_TASK': False, 'BINARY_SCHEME': 'NORM_vs_ALL'}


In [ ]:
# %% [markdown]
# ⚡ High-RAM Apple Silicon profile (32GB M1)

# %%
# Use GPU (Metal) + mixed precision; keep log-only (no artifacts).
CFG.update({
    "LOW_RAM": False,                # let tf.data AUTOTUNE & larger buffers
    "USE_FEATURE_CACHE": False,      # stays no-file by default
    "DEEP_CACHE_TO_DISK": False,     # caches in RAM (safe w/ 32GB)
    "SAVE_MIN_TABLE": False,
    "SAVE_ARTIFACTS": False,

    # Bigger training knobs
    "SEQ_LEN": 1000,                 # more temporal context (from 300)
    "DOWNSAMPLE_FACTOR": 2,          # retain more samples (was 5)
    "BATCH_SIZE": 32,                # increase batch
    "EPOCHS": 10,                    # extra epochs
    "DEEP_MAX_TRAIN_FRAC": 1.0,      # use full train set

    # TensorFlow acceleration
    "TF_MIXED_PRECISION": True,      # float16 on Apple GPU
    "TF_XLA": False,                 # keep False on mac unless tested
    "TF_INTRA_OP": 0,                # 0 → let TF decide
    "TF_INTER_OP": 0,
})

print("Applied High-RAM/Metal profile:", {
    k: CFG[k] for k in ["SEQ_LEN","DOWNSAMPLE_FACTOR","BATCH_SIZE","EPOCHS",
                        "LOW_RAM","TF_MIXED_PRECISION","TF_XLA","DEEP_MAX_TRAIN_FRAC"]
})


CFG updated for M1/low-RAM defaults.


In [ ]:
# %% [markdown]
# 🔧 Post-CFG tweak

# %%
CFG.update({"DEEP_HYBRID": False})
print("CFG updated.")


In [ ]:
# %% [markdown]
# 1) Setup & Reproducibility

# %%
# Ensure: pip install "tensorflow-macos==2.16.*" "tensorflow-metal==1.1.*" wfdb scikeras

import os, random, time, json, warnings, ast, sys, subprocess
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED)

def ts() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")
def log(msg: str): print(f"[{ts()}] {msg}")

try:
    from importlib.metadata import version
    log("Versions — pandas %s, numpy %s" % (version("pandas"), version("numpy")))
except Exception:
    pass

log("Seeds set.")


[2025-10-09 13:33:23] Versions — pandas 2.3.2, numpy 1.26.4
[2025-10-09 13:33:23] Seeds set.


In [4]:
# %% [markdown]
# 2) Paths & Sanity Checks (fixed explicit paths)

# %%
from pathlib import Path

# Your exact paths (as requested)
DB_CSV  = "dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv"
SCP_CSV = "dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv"

# Derive ROOT from DB_CSV so the rest of the notebook (records, caches) still works
ROOT = Path(DB_CSV).parent
RECORDS_DIR = ROOT / "records100"  # for filename_lr (100 Hz)

# Assertions
assert Path(DB_CSV).exists(),  f"Missing {DB_CSV}"
assert Path(SCP_CSV).exists(), f"Missing {SCP_CSV}"
assert RECORDS_DIR.exists(),   f"Missing {RECORDS_DIR} (100 Hz signals)"

# Quick WFDB sanity read
try:
    import wfdb
    sig0, meta0 = wfdb.rdsamp(str(RECORDS_DIR / "00000" / "00001_lr"))
    log(f"WFDB sanity OK: shape={sig0.shape}, leads={meta0.get('sig_name', [])}")
except ImportError as e:
    raise RuntimeError("Please install WFDB:  %pip install wfdb") from e
except Exception as e:
    log(f"WFDB sanity read failed (not fatal): {e}")



[2025-10-09 13:33:23] WFDB sanity OK: shape=(1000, 12), leads=['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


In [5]:
# %% [markdown]
# 3) Load CSVs (using explicit paths)

# %%
db  = pd.read_csv(DB_CSV)
scp_raw = pd.read_csv(SCP_CSV, encoding="utf-8-sig")
scp_raw.columns = [str(c).strip() for c in scp_raw.columns]

log(f"Loaded db from {DB_CSV}: shape={db.shape}")
log(f"Loaded scp_statements from {SCP_CSV}: shape={scp_raw.shape}")

# Table printouts
display(db.head(5))
display(scp_raw.head(5))


[2025-10-09 13:33:23] Loaded db from dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv: shape=(21799, 28)
[2025-10-09 13:33:23] Loaded scp_statements from dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv: shape=(71, 13)


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


,Unnamed: 0,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
0,NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
1,NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
2,DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
3,LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
4,NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [6]:
# %% [markdown]
# 4) Target Mapping (SCP → 5-class or 10-class)

# %%
def _extract_scp_keys(db_series, n=2000):
    keys = set()
    for s in db_series.head(min(n, len(db_series))):
        try:
            d = ast.literal_eval(s); keys.update(map(str, d.keys()))
        except Exception:
            pass
    return keys

known_keys = _extract_scp_keys(db["scp_codes"], n=2000)
candidates = [c for c in scp_raw.columns if scp_raw[c].dtype == "object"]
scores = {c: len(set(scp_raw[c].astype(str)).intersection(known_keys)) for c in candidates}
code_col = max(scores, key=scores.get) if scores and max(scores.values())>0 else scp_raw.columns[0]

scp_raw["code"] = scp_raw[code_col].astype(str)
scp = scp_raw.set_index("code", drop=False); scp.index.name = "code"

for req in ["diagnostic", "diagnostic_class", "diagnostic_subclass"]:
    if req not in scp.columns:
        raise ValueError(f"scp_statements.csv missing required column: '{req}'")

diag_col = scp["diagnostic"]
diag_mask_num = pd.to_numeric(diag_col, errors="coerce").fillna(0) > 0
diag_mask_str = diag_col.astype(str).str.strip().str.lower().isin(["1","1.0","true","t","yes","y","on"])
diag_mask = diag_mask_num | diag_mask_str

label_field = "diagnostic_class" if CFG["LABEL_MODE"] == "5class" else "diagnostic_subclass"
diag_only = scp.loc[diag_mask, ["diagnostic_class", "diagnostic_subclass"]].copy()
valid_codes = set(diag_only.index)

log(f"Diagnostic SCP codes detected: {len(valid_codes)} / {len(scp)}")
if len(valid_codes) == 0:
    raise RuntimeError("No diagnostic codes detected — check scp_statements.csv 'diagnostic'.")

def to_diag_label(scp_codes_dict, min_conf=0.0):
    try:
        items = [(str(k), float(v)) for k, v in scp_codes_dict.items()]
    except Exception:
        return None
    items = [(k, v) for k, v in items if k in valid_codes and v >= min_conf]
    if not items: return None
    lab_weights = {}
    for k, w in items:
        lab = diag_only.loc[k, label_field]
        lab_weights[lab] = lab_weights.get(lab, 0.0) + w
    return max(lab_weights.items(), key=lambda kv: kv[1])[0]


[2025-10-09 13:33:23] Diagnostic SCP codes detected: 44 / 71


In [7]:
# %% [markdown]
# 5) Feature Extraction (minimal table for deep) — log-only

# %%
from typing import Dict, Any
from joblib import Parallel, delayed

OUT_CSV = ROOT / "ptbxl_minimal_for_deep.csv"

META_FIELDS = [
    "ecg_id","patient_id","age","sex","device","recording_date","scp_codes","strat_fold"
]

def _one_row_min(r: dict) -> dict | None:
    row: Dict[str, Any] = {}
    for k in META_FIELDS:
        v = r.get(k, np.nan)
        if k == "recording_date":
            try: row[k] = pd.to_datetime(v).date().isoformat()
            except Exception: row[k] = np.nan
        else:
            row[k] = v
    try:
        scp_codes = ast.literal_eval(r.get("scp_codes", "{}"))
    except Exception:
        return None
    y = to_diag_label(scp_codes)
    if y is None: return None

    rec_dat   = ROOT / r[CFG["RECORD_FILE_COL"]]
    rec_noext = str(rec_dat)[:-4] if str(rec_dat).endswith(".dat") else str(rec_dat)
    row["record_path"] = rec_noext
    row["label"] = y
    return row

def build_minimal_table(db: pd.DataFrame, max_records=None) -> pd.DataFrame:
    rows = db.to_dict(orient="records")
    if max_records is not None:
        rows = rows[:max_records]
    results = Parallel(n_jobs=-1, prefer="threads")(delayed(_one_row_min)(r) for r in rows)
    results = [r for r in results if r is not None]
    if not results:
        raise RuntimeError("No rows produced — check file paths and mappings.")
    df = pd.DataFrame(results)
    log(f"Minimal deep table: n={len(df)}")
    return df

# Build without writing unless toggled
if OUT_CSV.exists() and CFG["USE_FEATURE_CACHE"] and CFG["DEEP_FAST_METADATA_ONLY"]:
    features_df = pd.read_csv(OUT_CSV)
    log(f"Using cached minimal table: {OUT_CSV} ({features_df.shape})")
else:
    features_df = build_minimal_table(db, CFG["MAX_RECORDS"])
    if CFG.get("SAVE_MIN_TABLE", False) and CFG["DEEP_FAST_METADATA_ONLY"]:
        OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
        features_df.to_csv(OUT_CSV, index=False)
        log(f"Saved minimal table → {OUT_CSV}")
    else:
        log("Minimal table generated (not saved to disk).")

# Table printouts
print("Shape:", features_df.shape)
display(features_df.head(10))
print("\nClass counts:")
display(features_df["label"].value_counts().rename_axis("label").to_frame("count"))


[2025-10-09 13:33:28] Minimal deep table: n=21388
[2025-10-09 13:33:28] Minimal table generated (not saved to disk).
Shape: (21388, 10)


,ecg_id,patient_id,age,sex,device,recording_date,scp_codes,strat_fold,record_path,label
0,1,15709.0,56.0,1,CS-12 E,1984-11-09,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",3,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
1,2,13243.0,19.0,0,CS-12 E,1984-11-14,"{'NORM': 80.0, 'SBRAD': 0.0}",2,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
2,3,20372.0,37.0,1,CS-12 E,1984-11-15,"{'NORM': 100.0, 'SR': 0.0}",5,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
3,4,17014.0,24.0,0,CS-12 E,1984-11-15,"{'NORM': 100.0, 'SR': 0.0}",3,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
4,5,17448.0,19.0,1,CS-12 E,1984-11-17,"{'NORM': 100.0, 'SR': 0.0}",4,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
5,6,19005.0,18.0,1,CS-12 E,1984-11-28,"{'NORM': 100.0, 'SR': 0.0}",4,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
6,7,16193.0,54.0,0,CS-12 E,1984-11-28,"{'NORM': 100.0, 'SR': 0.0}",7,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
7,8,11275.0,48.0,0,CS-12 E,1984-12-01,"{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}",9,dataset/ptb-xl-a-large-publicly-available-elec...,MI
8,9,18792.0,55.0,0,CS-12 E,1984-12-08,"{'NORM': 100.0, 'SR': 0.0}",10,dataset/ptb-xl-a-large-publicly-available-elec...,NORM
9,10,9456.0,22.0,1,CS-12 E,1984-12-12,"{'NORM': 100.0, 'SR': 0.0}",9,dataset/ptb-xl-a-large-publicly-available-elec...,NORM



Class counts:


,count
label,
NORM,9243
MI,4049
CD,3431
STTC,3360
HYP,1305


In [8]:
# %% [markdown]
# 6) EDA (basic) — table view

# %%
eda_na = features_df.isna().mean().sort_values(ascending=False).head(10).rename("NA_rate")
print("Top NA rates:")
display(eda_na.to_frame())
if "strat_fold" in features_df.columns:
    print("\nStrat_fold counts:")
    display(features_df["strat_fold"].value_counts().sort_index().rename_axis("strat_fold").to_frame("count"))


Top NA rates:


,NA_rate
ecg_id,0.0
patient_id,0.0
age,0.0
sex,0.0
device,0.0
recording_date,0.0
scp_codes,0.0
strat_fold,0.0
record_path,0.0
label,0.0



Strat_fold counts:


,count
strat_fold,
1,2139
2,2138
3,2150
4,2130
5,2135
6,2129
7,2134
8,2129
9,2146


In [9]:
# %% [markdown]
# 8) Leakage-safe Split (prefer PTB-XL strat_fold)

# %%
from sklearn.model_selection import GroupShuffleSplit

df = features_df.copy()
TARGET_COL = "label"

if "strat_fold" in df.columns and df["strat_fold"].notna().all():
    train_mask = df["strat_fold"].astype(int).isin(list(range(1,10)))
    test_mask  = df["strat_fold"].astype(int) == 10
    log("Using PTB-XL strat_fold split (1–9 → train, 10 → test).")
else:
    groups = df["patient_id"].values if "patient_id" in df.columns else np.arange(len(df))
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, test_idx = next(gss.split(df, df[TARGET_COL], groups))
    train_mask = df.index.isin(train_idx); test_mask = df.index.isin(test_idx)
    log("Using GroupShuffleSplit by patient_id.")

# Keep y_* as Series with original indices
y_train = df.loc[train_mask, TARGET_COL].copy()
X_train = df.loc[train_mask, ["record_path"]].copy()
y_test  = df.loc[test_mask,  TARGET_COL].copy()
X_test  = df.loc[test_mask,  ["record_path"]].copy()
log(f"Train/Test sizes: {len(X_train)} / {len(X_test)}")

# Table printouts
print("Train class counts:")
display(y_train.value_counts().sort_index().rename_axis("label").to_frame("count"))
print("\nTest class counts:")
display(y_test.value_counts().sort_index().rename_axis("label").to_frame("count"))


[2025-10-09 13:33:29] Using PTB-XL strat_fold split (1–9 → train, 10 → test).
[2025-10-09 13:33:29] Train/Test sizes: 19230 / 2158
Train class counts:


,count
label,
CD,3076
HYP,1182
MI,3644
NORM,8312
STTC,3016



Test class counts:


,count
label,
CD,355
HYP,123
MI,405
NORM,931
STTC,344


In [ ]:
# %% [markdown]
# 8b) Optional binary task switch (paper-style)

# %%
def _is_integer_series(s):
    try:
        return pd.api.types.is_integer_dtype(s.dtype)
    except Exception:
        return False

if bool(CFG.get("BINARY_TASK", False)):
    scheme = str(CFG.get("BINARY_SCHEME", "NORM_vs_ALL")).upper().replace("-", "_")

    y_train = pd.Series(y_train, index=y_train.index)
    y_test  = pd.Series(y_test,  index=y_test.index)

    if _is_integer_series(y_train) and _is_integer_series(y_test):
        y_train = (y_train.astype(int) != 0).astype(int)
        y_test  = (y_test.astype(int)  != 0).astype(int)
        CFG["POSITIVE_CLASS_NAME"] = "1"
        log("Binary task ON — labels already numeric; using 1 as positive.")
    else:
        ytr_str = y_train.astype(str)
        yte_str = y_test.astype(str)

        def _norm_vs_all():
            if (ytr_str == "NORM").any() or (yte_str == "NORM").any():
                y_tr_bin = (ytr_str != "NORM").astype(int)  # 1 = abnormal
                y_te_bin = (yte_str != "NORM").astype(int)
                pos_name = "ABNORMAL (not NORM)"
                msg = "NORM vs ALL (abnormal)"
            else:
                maj = ytr_str.value_counts().idxmax()
                y_tr_bin = (ytr_str != maj).astype(int)
                y_te_bin = (yte_str != maj).astype(int)
                pos_name = f"not {maj}"
                msg = f"NORM missing → '{maj}' vs ALL"
            return y_tr_bin, y_te_bin, pos_name, msg

        def _norm_vs_hyp():
            if (ytr_str == "HYP").any() or (yte_str == "HYP").any():
                y_tr_bin = (ytr_str == "HYP").astype(int)  # 1 = HYP
                y_te_bin = (yte_str == "HYP").astype(int)
                pos_name = "HYP"
                msg = "HYP vs not-HYP"
            else:
                y_tr_bin, y_te_bin, pos_name, base_msg = _norm_vs_all()
                msg = f"Fallback (HYP missing) → {base_msg}"
            return y_tr_bin, y_te_bin, pos_name, msg

        if scheme in {"NORM_VS_HYP", "NORM_VS_HYPERTENSION", "NORM_VS_HYPER"}:
            y_train, y_test, pos_name, msg = _norm_vs_hyp()
        else:
            y_train, y_test, pos_name, msg = _norm_vs_all()

        CFG["POSITIVE_CLASS_NAME"] = pos_name
        log(f"Binary task ON — {msg} | positive class = {pos_name}")
else:
    log("Binary task OFF — keeping multiclass labels.")


[2025-10-09 13:33:29] Binary task OFF — keeping multiclass labels.


: 

In [ ]:
# %% [markdown]
# 🧩 TensorFlow setup (M1-safe) + Speed dials

# %%
_cfg = globals().get("CFG", {})
RUN_TF = any([
    bool(_cfg.get("RUN_TF_CNN", False)),
    bool(_cfg.get("RUN_TF_RNN", False)),
    bool(_cfg.get("RUN_TF_LSTM", False)),
    bool(_cfg.get("RUN_TF_ANN", False)),
])

if not RUN_TF:
    print("[TF] Deep models are OFF — skipping TensorFlow setup.")
else:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU
    os.environ["ZE_AFFINITY_MASK"] = ""
    os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

    import tensorflow as tf
    try:
        tf.config.optimizer.set_jit(bool(_cfg.get("TF_XLA", False)))
    except Exception:
        pass

    intra = int(_cfg.get("TF_INTRA_OP", 1))
    inter = int(_cfg.get("TF_INTER_OP", 1))
    tf.config.threading.set_intra_op_parallelism_threads(intra)
    tf.config.threading.set_inter_op_parallelism_threads(inter)

    import numpy as _np
    print(f"[TF] v{tf.__version__} — backend: CPU | threads: intra={intra}, inter={inter} | numpy={_np.__version__}")
    _ = tf.linalg.matmul(tf.random.uniform((64,64)), tf.random.uniform((64,64)))
    print("[TF] Matmul OK.")


In [ ]:
# %% [markdown]
# 10b) Deep Models (CNN, RNN, LSTM, ANN) — streaming raw 12-lead sequences

# %%
# === Deep section (STREAMING, low-RAM) ===
import os, gc, time, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

_missing = [v for v in ["CFG","df","X_train","X_test","y_train","y_test"] if v not in globals()]
if _missing:
    raise RuntimeError(f"Missing variables: {_missing}. Run Sections 0–8 first.")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import wfdb

ART_DIR = Path(CFG["ART_DIR"]); ART_DIR.mkdir(parents=True, exist_ok=True)

# ---- Sizes & knobs ----
SEQ_LEN   = int(CFG.get("SEQ_LEN", 300))
DS_FACTOR = int(CFG.get("DOWNSAMPLE_FACTOR", 5))
T_EFF     = max(1, SEQ_LEN // max(1, DS_FACTOR))  # effective time steps after downsample
BATCH     = int(CFG.get("BATCH_SIZE", 16))
EPOCHS    = int(CFG.get("EPOCHS", 6))
VAL_FRAC  = 0.10
USE_FILE_CACHE = bool(CFG.get("DEEP_CACHE_TO_DISK", True))
MAX_FRAC  = float(CFG.get("DEEP_MAX_TRAIN_FRAC", 0.25))

print(f"[Deep/Streaming] target T={T_EFF} (SEQ_LEN={SEQ_LEN}, DS={DS_FACTOR}), BATCH={BATCH}")

# ---- Build label encoder on CURRENT split ----
tr_idx, te_idx = X_train.index, X_test.index
if 0 < MAX_FRAC < 1.0:
    tr_idx = pd.Series(tr_idx).sample(frac=MAX_FRAC, random_state=CFG["SEED"]).tolist()
    print(f"[Deep/Streaming] Using {len(tr_idx)} train rows (frac={MAX_FRAC})")

train_paths = df.loc[tr_idx, "record_path"].astype(str).values
test_paths  = df.loc[te_idx, "record_path"].astype(str).values

if len(train_paths) == 0 or len(test_paths) == 0:
    raise RuntimeError("No paths in train/test split. Reduce DEEP_MAX_TRAIN_FRAC or disable FAST_RUN.")

le = LabelEncoder()
y_tr = le.fit_transform(pd.Series(y_train).loc[tr_idx].values)
y_te = le.transform(pd.Series(y_test).loc[te_idx].values)
n_classes = len(le.classes_)
is_binary = (n_classes == 2)
print(f"[Deep/Streaming] Classes={list(le.classes_)}")

# ---- robust loader enforcing consistent 12-lead layout when available ----
STD_LEADS = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

def _py_load(rec_path: bytes, T=T_EFF, factor=DS_FACTOR):
    path = rec_path.decode("utf-8")
    x, meta = wfdb.rdsamp(path)          # (time, channels)
    x = x.astype("float32")
    try:
        sig_names = meta.get("sig_name", [])
        if len(sig_names) >= 12 and all(s in sig_names for s in STD_LEADS):
            idx = [sig_names.index(s) for s in STD_LEADS]
            x = x[:, idx]
    except Exception:
        pass

    if factor > 1:
        x = x[::factor]
    if x.shape[0] >= T:
        x = x[:T, :]
    else:
        pad = np.zeros((T - x.shape[0], x.shape[1]), dtype="float32")
        x = np.vstack([x, pad])
    return x

def _tf_load(path):
    x = tf.numpy_function(_py_load, [path], Tout=tf.float32)
    x.set_shape([T_EFF, None])   # time x channels; channel inferred first batch
    return x

# discover channels safely
_sample = None
try:
    _sample = _py_load(train_paths[0])
except Exception:
    for p in train_paths[1:6]:
        try:
            _sample = _py_load(p); break
        except Exception:
            continue
if _sample is None:
    raise RuntimeError("Unable to infer channel count from training samples.")
N_CH = int(_sample.shape[1])
print(f"[Deep/Streaming] Detected leads (channels): {N_CH}")

def _tf_load_fixed(path):
    x = tf.numpy_function(_py_load, [path], Tout=tf.float32)
    x.set_shape([T_EFF, N_CH])
    return x

# ---- Make streaming datasets ----
cache_dir = ART_DIR / "tfcache"; cache_dir.mkdir(parents=True, exist_ok=True)

def make_stream_ds(paths, labels=None, training=True, tag="train"):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)) if labels is not None \
         else tf.data.Dataset.from_tensor_slices(paths)
    if labels is not None:
        ds = ds.shuffle(min(len(paths), 10000), seed=CFG["SEED"], reshuffle_each_iteration=True)
        ds = ds.map(lambda p, y: (_tf_load_fixed(p), y),
                    num_parallel_calls=1 if CFG["LOW_RAM"] else tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda p: _tf_load_fixed(p),
                    num_parallel_calls=1 if CFG["LOW_RAM"] else tf.data.AUTOTUNE)

    if USE_FILE_CACHE:
        key = f"{tag}_T{T_EFF}_C{N_CH}"
        ds = ds.cache(str(cache_dir / f"{key}.cache"))
    else:
        ds = ds.cache()

    ds = ds.batch(BATCH, drop_remainder=False).prefetch(tf.data.AUTOTUNE if not CFG["LOW_RAM"] else 1)
    return ds

# Build train/val split by slicing
n_val = max(1, int(len(y_tr) * VAL_FRAC))
train_ds = make_stream_ds(train_paths[n_val:], y_tr[n_val:], training=True,  tag="train")
val_ds   = make_stream_ds(train_paths[:n_val],  y_tr[:n_val],  training=False, tag="val")
test_ds  = make_stream_ds(test_paths,          None,          training=False, tag="test")

input_shape = (T_EFF, N_CH)

# ---- Output heads / compile helper ----
def _out_and_loss(x, n_classes):
    if n_classes == 2:
        return layers.Dense(1, activation="sigmoid")(x), "binary_crossentropy"
    return layers.Dense(n_classes, activation="softmax")(x), "sparse_categorical_crossentropy"

def _compile(m, loss, lr=3e-4):
    opt = keras.optimizers.Adam(lr)
    try:
        from tensorflow.keras import mixed_precision
        if mixed_precision.global_policy().name == "mixed_float16":
            opt = mixed_precision.LossScaleOptimizer(opt)
    except Exception:
        pass
    try:
        m.compile(optimizer=opt, loss=loss, metrics=["accuracy"], jit_compile=False)
    except TypeError:
        m.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return m

# ---- Models (kept modest) ----
def build_cnn(shape, n_classes):
    inp = keras.Input(shape=shape)
    x = layers.AveragePooling1D(2)(inp)
    x = layers.Conv1D(24, 7, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(48, 5, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(96, 3, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(96, activation="relu")(x)
    out, loss = _out_and_loss(x, n_classes)
    return _compile(keras.Model(inp, out), loss, lr=3e-4)

def build_rnn_simple(shape, n_classes):
    inp = keras.Input(shape=shape)
    x = layers.AveragePooling1D(2)(inp)
    x = layers.SimpleRNN(96, return_sequences=True)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.SimpleRNN(48)(x)
    x = layers.Dense(64, activation="relu")(x)
    out, loss = _out_and_loss(x, n_classes)
    return _compile(keras.Model(inp, out), loss, lr=3e-4)

def build_lstm(shape, n_classes):
    inp = keras.Input(shape=shape)
    x = layers.AveragePooling1D(2)(inp)
    x = layers.LSTM(96, return_sequences=True)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.LSTM(48)(x)
    x = layers.Dense(64, activation="relu")(x)
    out, loss = _out_and_loss(x, n_classes)
    return _compile(keras.Model(inp, out), loss, lr=3e-4)

def build_ann(shape, n_classes):
    inp = keras.Input(shape=shape)
    x = layers.GlobalAveragePooling1D()(inp)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation="relu")(x)
    out, loss = _out_and_loss(x, n_classes)
    return _compile(keras.Model(inp, out), loss, lr=3e-4)

# ---- Trainer ----
def train_model(name, builder, epochs=EPOCHS):
    keras.backend.clear_session()
    model = builder(input_shape, n_classes)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, min_delta=1e-3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
    ]
    print(f"[Deep/Streaming] Training {name}: epochs={epochs}, batch={BATCH}")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=1, callbacks=callbacks)
    return model

# ---- Run selected models ----
deep_models = {}
if bool(CFG.get("RUN_TF_CNN", True)):
    cnn_model = train_model("1D CNN", build_cnn, epochs=min(CFG["EPOCHS"], 10))
    deep_models["CNN (Keras)"] = cnn_model
if bool(CFG.get("RUN_TF_RNN", True)):
    rnn_model = train_model("SimpleRNN", build_rnn_simple, epochs=min(CFG["EPOCHS"], 10))
    deep_models["RNN (Keras)"] = rnn_model
if bool(CFG.get("RUN_TF_LSTM", True)):
    lstm_model = train_model("LSTM", build_lstm, epochs=min(CFG["EPOCHS"], 10))
    deep_models["LSTM (Keras)"] = lstm_model
if bool(CFG.get("RUN_TF_ANN", True)):
    ann_model = train_model("ANN (pooled)", build_ann, epochs=min(CFG["EPOCHS"], 10))
    deep_models["ANN (Keras)"] = ann_model

# ---- sklearn-like adapter that works with PATHS (streaming) ----
class KerasAdapter:
    def __init__(self, model, label_encoder, is_binary: bool, batch=BATCH):
        self.model = model; self.le = label_encoder; self.is_binary = is_binary
        self.batch = batch; self.classes_ = self.le.classes_

    def _make_predict_ds(self, paths):
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(lambda p: _tf_load_fixed(p), num_parallel_calls=1 if CFG["LOW_RAM"] else tf.data.AUTOTUNE)
        if CFG["DEEP_CACHE_TO_DISK"]:
            key = f"predict_T{T_EFF}_C{N_CH}"
            ds = ds.cache(str((ART_DIR / "tfcache" / f"{key}.cache")))
        else:
            ds = ds.cache()
        return ds.batch(self.batch, drop_remainder=False).prefetch(tf.data.AUTOTUNE if not CFG["LOW_RAM"] else 1)

    def predict(self, paths):
        ds = self._make_predict_ds(paths)
        if self.is_binary:
            p1 = self.model.predict(ds, verbose=0).ravel()
            y_idx = (p1 >= 0.5).astype(int)
        else:
            proba = self.model.predict(ds, verbose=0)
            y_idx = proba.argmax(axis=1)
        return self.le.inverse_transform(y_idx)

    def predict_proba(self, paths):
        ds = self._make_predict_ds(paths)
        proba = self.model.predict(ds, verbose=0)
        if self.is_binary:
            p1 = proba.ravel(); p0 = 1.0 - p1
            return np.vstack([p0, p1]).T.astype("float32")
        return proba.astype("float32")

# Register adapters
adapters = {name: KerasAdapter(mdl, le, is_binary, batch=BATCH) for name, mdl in deep_models.items()}
deep_models = adapters
DEEP_TEST_PATHS = test_paths

print("[Deep/Streaming] Models ready:", list(deep_models.keys()))
gc.collect()


In [ ]:
# %% [markdown]
# 10c) (Optional) GridSearchCV with SciKeras (small subset; ANN/CNN/LSTM) — 5 folds

# %%
RUN_GRID = False  # set True to run
N_FOLDS = 5       # <<< using 5-fold GroupKFold

if RUN_GRID:
    # !pip install -q scikeras[tensorflow]
    import numpy as np, wfdb, tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import GroupKFold, GridSearchCV
    from scikeras.wrappers import KerasClassifier

    MODEL_KIND = "ann"    # "ann"|"cnn"|"lstm"
    SUBSET_MAX = 2000
    T = T_EFF; FACTOR = DS_FACTOR

    rng = np.random.default_rng(CFG["SEED"])
    tr_idx_np = np.array(list(X_train.index))
    if len(tr_idx_np) > SUBSET_MAX:
        tr_idx_np = rng.choice(tr_idx_np, size=SUBSET_MAX, replace=False)

    sub_paths  = df.loc[tr_idx_np, "record_path"].astype(str).values
    sub_labels = pd.Series(y_train).loc[tr_idx_np].astype(str).values
    sub_groups = df.loc[tr_idx_np, "patient_id"].values if "patient_id" in df.columns else np.arange(len(tr_idx_np))

    STD_LEADS = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
    def load_one(path, T=T, factor=FACTOR):
        x, meta = wfdb.rdsamp(path); x = x.astype("float32")
        try:
            sig_names = meta.get("sig_name", [])
            if len(sig_names) >= 12 and all(s in sig_names for s in STD_LEADS):
                idx = [sig_names.index(s) for s in STD_LEADS]
                x = x[:, idx]
        except Exception: pass
        if factor > 1: x = x[::factor]
        if x.shape[0] >= T: x = x[:T, :]
        else:
            pad = np.zeros((T - x.shape[0], x.shape[1]), dtype="float32")
            x = np.vstack([x, pad])
        return x

    _sample = None
    for p in sub_paths[:10]:
        try: _sample = load_one(p); break
        except Exception: continue
    if _sample is None: raise RuntimeError("Could not read any sample for GridSearch.")
    N_CH = _sample.shape[1]

    X_sub = np.zeros((len(sub_paths), T, N_CH), dtype="float32")
    bad=[]
    for i,p in enumerate(sub_paths):
        try: X_sub[i] = load_one(p)
        except Exception: bad.append(i)
    if bad:
        keep = np.setdiff1d(np.arange(len(sub_paths)), np.array(bad))
        X_sub = X_sub[keep]; sub_labels = sub_labels[keep]; sub_groups = sub_groups[keep]

    le2 = LabelEncoder(); y_sub = le2.fit_transform(sub_labels)
    n_classes2 = len(le2.classes_

)
    def make_ann(n_units=128, dropout=0.2, lr=3e-4):
        inp = keras.Input(shape=(T, N_CH))
        x = layers.GlobalAveragePooling1D()(inp)
        x = layers.Dense(n_units, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(n_units, activation="relu")(x)
        if n_classes2 == 2:
            out = layers.Dense(1, activation="sigmoid")(x); loss = "binary_crossentropy"
        else:
            out = layers.Dense(n_classes2, activation="softmax")(x); loss = "sparse_categorical_crossentropy"
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
        return model

    def make_cnn(nf=32, lr=3e-4, dropout=0.2):
        inp = keras.Input(shape=(T, N_CH))
        x = layers.Conv1D(nf, 7, padding="same", activation="relu")(inp)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(nf*2, 5, padding="same", activation="relu")(x)
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dropout(dropout)(x)
        if n_classes2 == 2:
            out = layers.Dense(1, activation="sigmoid")(x); loss = "binary_crossentropy"
        else:
            out = layers.Dense(n_classes2, activation="softmax")(x); loss = "sparse_categorical_crossentropy"
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
        return model

    def make_lstm(n_units=64, lr=3e-4, dropout=0.2):
        inp = keras.Input(shape=(T, N_CH))
        x = layers.LSTM(n_units, return_sequences=False, dropout=dropout)(inp)
        if n_classes2 == 2:
            out = layers.Dense(1, activation="sigmoid")(x); loss = "binary_crossentropy"
        else:
            out = layers.Dense(n_classes2, activation="softmax")(x); loss = "sparse_categorical_crossentropy"
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
        return model

    builder = {"ann": make_ann, "cnn": make_cnn, "lstm": make_lstm}[MODEL_KIND]
    from scikeras.wrappers import KerasClassifier
    clf = KerasClassifier(model=builder, epochs=6, batch_size=16, verbose=0, random_state=CFG["SEED"])

    if MODEL_KIND == "ann":
        param_grid = {"model__n_units":[64,128], "model__dropout":[0.1,0.3], "model__lr":[3e-4,1e-3],
                      "epochs":[4,6], "batch_size":[8,16]}
    elif MODEL_KIND == "cnn":
        param_grid = {"model__nf":[16,32], "model__dropout":[0.1,0.3], "model__lr":[3e-4,1e-3],
                      "epochs":[4,6], "batch_size":[8,16]}
    else:
        param_grid = {"model__n_units":[32,64], "model__dropout":[0.1,0.3], "model__lr":[3e-4,1e-3],
                      "epochs":[4,6], "batch_size":[8,16]}

    # 5-fold GroupKFold to avoid patient leakage
    cv = GroupKFold(n_splits=N_FOLDS)
    scoring = "f1_macro" if n_classes2 > 2 else "f1"

    grid = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv.split(X_sub, y_sub, groups=sub_groups),
        n_jobs=1, verbose=2, return_train_score=False
    )

    print(f"[GridSearch] folds={N_FOLDS}, kind={MODEL_KIND}, classes={list(le2.classes_)}, X={X_sub.shape}")
    grid.fit(X_sub, y_sub)
    print("\nBest params:", grid.best_params_)
    print("Best CV score:", grid.best_score_)


In [ ]:
# %% [markdown]
# 11) Evaluation — metrics, confusion matrix, reliability (deep-only)

# %%
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, \
                            confusion_matrix, roc_auc_score
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

# Dummy baseline (reference only)
from sklearn.pipeline import Pipeline as SKPipe
dummy_pipe = SKPipe([("clf", DummyClassifier(strategy="most_frequent"))])
dummy_pipe.fit(np.zeros((len(y_train), 1)), y_train)

def _scores_or_none(model, X):
    if hasattr(model, "predict_proba"):
        try: return model.predict_proba(X), getattr(model, "classes_", None), "proba"
        except Exception: pass
    return None, None, None

def _align_scores_to_classes(scores, est_classes, target_classes):
    if scores is None or est_classes is None: return scores
    est_classes = np.asarray(est_classes); target_classes = np.asarray(target_classes)
    pos = {c: i for i, c in enumerate(est_classes)}
    idx = []
    for c in target_classes:
        if c not in pos: return None
        idx.append(pos[c])
    return scores[:, idx]

def eval_model(name, model, X_in, y_true, show_cm=True):
    y_pred = model.predict(X_in)
    y_scores, est_classes, _ = _scores_or_none(model, X_in)
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc_ovr_macro": np.nan
    }
    if y_scores is not None:
        classes = np.unique(y_true)
        if len(classes) > 1:
            from sklearn.preprocessing import label_binarize
            y_bin = label_binarize(y_true, classes=classes)
            y_scores_aligned = _align_scores_to_classes(y_scores, est_classes, classes)
            if (y_scores_aligned is not None):
                try:
                    metrics["roc_auc_ovr_macro"] = float(
                        roc_auc_score(y_bin, y_scores_aligned, average="macro", multi_class="ovr")
                    )
                except Exception: pass

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    if show_cm:
        labels = np.unique(y_true)
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        plt.figure(); plt.imshow(cm, interpolation="nearest"); plt.title(f"{name} — Confusion Matrix"); plt.colorbar()
        ticks = np.arange(len(labels))
        plt.xticks(ticks, labels, rotation=45); plt.yticks(ticks, labels)
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()
    return metrics

# Build jobs: dummy uses zeros; deep models use PATHS
eval_jobs = [("Dummy (most_frequent)", dummy_pipe, np.zeros((len(y_test),1)))]
for n, m in deep_models.items():
    eval_jobs.append((n, m, DEEP_TEST_PATHS))

results = [eval_model(name, mdl, Xin, y_test) for (name, mdl, Xin) in eval_jobs]
res_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)

# Table printout of metrics
print("\nEvaluation summary (sorted by weighted F1):")
display(res_df)

# Reliability (aligned)
plt.figure()
for (name, mdl, Xin) in eval_jobs:
    if name.startswith("Dummy") or not hasattr(mdl, "predict_proba"): 
        continue
    try:
        proba = mdl.predict_proba(DEEP_TEST_PATHS)
        if hasattr(mdl, "classes_") and mdl.classes_ is not None:
            cls = np.array(list(mdl.classes_))
            pos = {c: i for i, c in enumerate(cls)}
            y_true_arr = pd.Series(y_test).values
            y_enc = np.array([pos.get(lbl, -1) for lbl in y_true_arr])
            ok = y_enc >= 0
            proba, y_enc = proba[ok], y_enc[ok]
        else:
            cls = np.unique(y_test)
            if proba.shape[1] != len(cls): 
                continue
            y_true_arr = pd.Series(y_test).values
            y_enc = np.searchsorted(cls, y_true_arr)

        conf  = proba.max(axis=1)
        preds = proba.argmax(axis=1)
        correct = (preds == y_enc).astype(int)

        edges = np.linspace(0,1,11)
        idx = np.digitize(conf, edges)-1
        bin_conf, bin_acc = [], []
        for b in range(10):
            m = idx==b
            if m.sum():
                bin_conf.append(conf[m].mean()); bin_acc.append(correct[m].mean())
        if bin_conf:
            plt.plot(bin_conf, bin_acc, marker="o", label=name)
    except Exception:
        continue

plt.plot([0,1],[0,1],"--", alpha=.6)
plt.xlabel("Confidence"); plt.ylabel("Accuracy"); plt.title("Reliability Diagram"); plt.legend(); plt.show()


In [ ]:
# %% [markdown]
# 12) Interpretation (note)

# %%
print("Classical model feature importances removed. For deep models, use per-class metrics or saliency if needed.")


In [ ]:
# %% [markdown]
# 15) Subgroup Fairness (sex, age bins, device/site) — table view

# %%
from sklearn.metrics import f1_score
import warnings
import numpy as np
import pandas as pd

best_name = None
try:
    best_name = res_df.iloc[0]["model"]
except Exception:
    if len(deep_models): best_name = list(deep_models.keys())[0]

if best_name is None:
    print("No deep model found for fairness check.")
else:
    mdl = deep_models[best_name]
    test_index = X_test.index
    meta_cols_eval = ["ecg_id","patient_id","age","sex","device","recording_date","scp_codes","strat_fold"]
    test_meta = df.loc[test_index, meta_cols_eval].copy()
    test_meta["y_true"]  = pd.Series(y_test).values
    y_pred = mdl.predict(DEEP_TEST_PATHS)
    test_meta["pred_best"] = y_pred

    # Tables
    print(f"\nWeighted F1 by sex ({best_name}):")
    rows = []
    for g, gdf in test_meta.groupby("sex"):
        if len(gdf):
            f1w = f1_score(gdf["y_true"], gdf["pred_best"], average="weighted", zero_division=0)
            rows.append({"sex": g, "n": len(gdf), "f1_weighted": f1w})
    display(pd.DataFrame(rows).sort_values("sex"))

    bins = [-np.inf, 40, 65, np.inf]; labels_bins = ["<40", "40–65", "65+"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        test_meta["age_bin"] = pd.cut(test_meta["age"], bins=bins, labels=labels_bins)

    print(f"\nWeighted F1 by age_bin ({best_name}):")
    rows = []
    for g, gdf in test_meta.groupby("age_bin"):
        if len(gdf):
            f1w = f1_score(gdf["y_true"], gdf["pred_best"], average="weighted", zero_division=0)
            rows.append({"age_bin": str(g), "n": len(gdf), "f1_weighted": f1w})
    display(pd.DataFrame(rows).sort_values("age_bin"))

    if "device" in test_meta.columns:
        print(f"\nWeighted F1 by device ({best_name}):")
        rows = []
        for g, gdf in test_meta.groupby("device"):
            if len(gdf):
                f1w = f1_score(gdf["y_true"], gdf["pred_best"], average="weighted", zero_division=0)
                rows.append({"device": g, "n": len(gdf), "f1_weighted": f1w})
        display(pd.DataFrame(rows).sort_values("device"))


In [ ]:
# %% [markdown]
# 16) Log-only Reporting (no files written)

# %%
from pprint import pprint

print("\n" + "="*70)
print("RUN SUMMARY (log-only, no artifacts saved)")
print("="*70)

# 1) Config snapshot
print("\n[Config]")
cfg_view = {k: CFG[k] for k in [
    "LABEL_MODE","SEQ_LEN","DOWNSAMPLE_FACTOR","BATCH_SIZE","EPOCHS",
    "DEEP_MAX_TRAIN_FRAC","LOW_RAM","USE_FEATURE_CACHE","DEEP_CACHE_TO_DISK",
    "RUN_TF_CNN","RUN_TF_RNN","RUN_TF_LSTM","RUN_TF_ANN","SAVE_MIN_TABLE","SAVE_ARTIFACTS"
] if k in CFG}
pprint(cfg_view)

# 2) Classes and split sizes
print("\n[Data]")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
try:
    print("\nTrain class counts:")
    display(pd.Series(y_train).value_counts().sort_index().rename_axis("label").to_frame("count"))
    print("\nTest class counts:")
    display(pd.Series(y_test).value_counts().sort_index().rename_axis("label").to_frame("count"))
except Exception as e:
    print(f"(Counts unavailable: {e})")

# 3) Trained models
print("\n[Models trained]")
print(", ".join(deep_models.keys()) if len(deep_models) else "None")

# 4) Evaluation table
try:
    print("\n[Evaluation — sorted by weighted F1]")
    display(res_df.sort_values("f1_weighted", ascending=False))
except Exception as e:
    print(f"(No evaluation table available: {e})")

# 5) Best model name
try:
    best_name = res_df.iloc[0]["model"]
    print(f"\nBest model: {best_name}")
except Exception:
    if len(deep_models):
        print(f"\nBest model (fallback): {list(deep_models.keys())[0]}")
    else:
        print("\nBest model: None")

print("\n(As requested: no CSVs, models, or ZIPs were written.)")


In [ ]:
# %% [markdown]
# 17) Appendix — Raw ECG viewer (first misclassified example)

# %%
try:
    import wfdb
    best_name = None
    try:
        best_name = res_df.iloc[0]["model"]
    except Exception:
        if len(deep_models): best_name = list(deep_models.keys())[0]
    if best_name is None:
        log("No deep model available for appendix plot.")
    else:
        mdl = deep_models[best_name]
        y_pred = mdl.predict(DEEP_TEST_PATHS)
        mis_mask = (pd.Series(y_test).values != pd.Series(y_pred).values)
        if mis_mask.any():
            mis_pos = np.where(mis_mask)[0][0]
            idx = X_test.index[mis_pos]
            rec_path = df.loc[idx, "record_path"]
            sig, meta = wfdb.rdsamp(rec_path)
            import matplotlib.pyplot as plt
            plt.figure(figsize=(10,4))
            plt.plot(sig[:,0])
            plt.title(f"Misclassified example — lead 0\n{rec_path}")
            plt.xlabel("Time (samples @100 Hz)"); plt.ylabel("Amplitude")
            plt.tight_layout(); plt.show()
        else:
            log("No misclassifications to display.")
except Exception as e:
    log(f"Raw viewer skipped: {e}")
